In [2]:
# 1. INSTALLS
!apt-get install -y tesseract-ocr
!pip install -q easyocr moviepy openai-whisper google-generativeai scikit-learn google-cloud-bigquery

# 2. IMPORTS
import os
import re
import easyocr
import whisper
import numpy as np
import google.generativeai as genai
from google.cloud import bigquery
from google.colab import userdata, files
from moviepy.editor import VideoFileClip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

# 3. AUTHENTICATION & ENGINES
# Set your Gemini API Key in Colab Secrets first
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/credentials.json"
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# Initialize Tools
# Use the newer model name we discovered was valid for you
model = genai.GenerativeModel('models/gemini-2.5-flash')
reader = easyocr.Reader(['en'], gpu=True) # Set to True once you enable T4 GPU
audio_model = whisper.load_model("tiny", device="cuda") # Set to "cuda" once you enable T4 GPU
bq_client = bigquery.Client()

print("✅ System Ready: OCR, Whisper, BigQuery, and Gemini are linked.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
✅ System Ready: OCR, Whisper, BigQuery, and Gemini are linked.


In [3]:
class OmniGuardAuditor:
    def __init__(self, dataset_id, table_id):
        self.table_path = f"{bq_client.project}.{dataset_id}.{table_id}"
        self.reference_corpus = self._fetch_from_bigquery()

    def _fetch_from_bigquery(self):
        """Fetches patent/IP text from your BigQuery table."""
        try:
            query = f"SELECT * FROM `{self.table_path}` LIMIT 200"
            df = bq_client.query(query).to_dataframe()
            # Automatically find the column containing the text
            text_col = [c for c in df.columns if any(x in c.lower() for x in ['text', 'script', 'abstract'])][0]
            return df[text_col].astype(str).tolist()
        except Exception as e:
            print(f"⚠️ BigQuery Fetch Error: {e}. Check table names.")
            return []

    def analyze_plagiarism(self, input_text):
        """TF-IDF Cosine Similarity for Script/Patent Theft."""
        if not self.reference_corpus or not input_text.strip():
            return 0.0
        tfidf = TfidfVectorizer().fit_transform([input_text] + self.reference_corpus)
        vectors = tfidf.toarray()
        # Input (0) vs Reference Corpus (1:)
        scores = cosine_similarity([vectors[0]], vectors[1:]).flatten()
        return round(float(np.max(scores)) * 100, 2)

    def analyze_compliance(self, text):
        """Context-based Regulatory Audit (Politics, Substance, Weight Loss)."""
        prompt = f"""
        Act as an IPR and Compliance Auditor. Analyze this content:
        "{text}"

        CHECK FOR:
        1. POLITICS: Campaign intent without 'Paid for by' disclaimers.
        2. SUBSTANCES: Promotion of Alcohol, Tobacco, or Vaping.
        3. WEIGHT LOSS: Scam detection (instant results) and missing medical warnings.
        4. SCRIPT THEFT: Is the structure similar to a known published work?
        """
        response = model.generate_content(prompt)
        return response.text

    def audit_file(self, file_path):
        extracted_text = ""
        if file_path.lower().endswith(('.mp4', '.mov')):
            # Audio Transcription
            print("🎙️ Transcribing Audio...")
            extracted_text += audio_model.transcribe(file_path)['text'] + " "
            # Frame Extraction for OCR
            print("🖼️ Scanning Video Frames...")
            clip = VideoFileClip(file_path)
            clip.save_frame("tmp.jpg", t=clip.duration/2)
            extracted_text += " ".join(reader.readtext("tmp.jpg", detail=0))
        else:
            extracted_text = " ".join(reader.readtext(file_path, detail=0))

        plag_score = self.analyze_plagiarism(extracted_text)
        compliance_report = self.analyze_compliance(extracted_text)
        return plag_score, compliance_report

In [6]:
# UPDATE THESE with your discovered BigQuery info
auditor = OmniGuardAuditor(dataset_id="legal_data", table_id="patent_corpus")

print("📤 Upload Video/Image for Audit:")
uploaded = files.upload()

for name in uploaded.keys():
    print(f"\n--- 🔎 ANALYZING: {name} ---")
    plag, report = auditor.audit_file(name)

    print(f"\n📊 IPR PLAGIARISM RISK: {plag}%")
    print(f"⚖️ COMPLIANCE REPORT:\n{report}")

⚠️ BigQuery Fetch Error: 404 Not found: Table patent-checker-app:legal_data.patent_corpus was not found in location US; reason: notFound, message: Not found: Table patent-checker-app:legal_data.patent_corpus was not found in location US

Location: US
Job ID: 9226ad8c-28cd-407f-808b-d96607b35ad6
. Check table names.
📤 Upload Video/Image for Audit:


Saving alcohol+weight.mp4 to alcohol+weight.mp4

--- 🔎 ANALYZING: alcohol+weight.mp4 ---
🎙️ Transcribing Audio...
🖼️ Scanning Video Frames...

📊 IPR PLAGIARISM RISK: 0.0%
⚖️ COMPLIANCE REPORT:
As an IPR and Compliance Auditor, I have analyzed the provided content:

"激的な変化をボレくださいよ新しいプログラムでねあなたもたった32中まで変われますねさあね今すぐお電話をお出そう {R AFTERI J5 "5 Veo"

**Analysis:**

First, let's break down the intelligible Japanese text and then address the garbled portion.

**Japanese Text Translation & Interpretation:**
*   "激的な変化をボレくださいよ新しいプログラムでね" - "Please allow for dramatic change with the new program, you know." (Note: "ボレください" (bore-kudasai) is an unusual/informal phrasing, likely intended to mean "experience," "achieve," or "embrace" dramatic change.)
*   "あなたもたった32中まで変われますね" - "You too can change in just 32 [something] (e.g., within 32 days/weeks/etc.)." (The unit "中" (naka) is ambiguous, meaning "within" or "in the middle of," but without a specific timeframe unit like days or weeks, it's vague.)
*  